# 01 - Data Ingestion Pipeline
This notebook outlines the extraction phase of the Bluestock Mutual Fund Analytics capstone using the manager's official datasets.
It details:
1. Setting up paths to raw files
2. Loading raw CSV data into pandas
3. Validating basic data integrity (such as AMFI codes)
4. Mocking/fetching the live AMFI API (api.mfapi.in)


In [ ]:
import pandas as pd
import requests
import pathlib
import json

# 1. Setup paths
base_dir = pathlib.Path('D:/New folder/bluestock_mf_capstone')
raw_dir = base_dir / 'data/raw'
print('Raw data directory:', raw_dir.resolve())


Raw data directory: D:\New folder\bluestock_mf_capstone\data\raw


## Loading Raw CSV Files and Verifying Dimensions


In [2]:
df_fund = pd.read_csv(raw_dir / '01_fund_master.csv')
df_nav = pd.read_csv(raw_dir / '02_nav_history.csv')
df_bench = pd.read_csv(raw_dir / '10_benchmark_indices.csv')

print(f'Fund Master records: {df_fund.shape}')
print(f'NAV History records: {df_nav.shape}')
print(f'Benchmark Data records: {df_bench.shape}')

print("\n--- Fund Master Sample ---")
display(df_fund.head())


Fund Master records: (40, 15)
NAV History records: (46000, 3)
Benchmark Data records: (8050, 3)

--- Fund Master Sample ---


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


## Basic Validation: AMFI Code Check
AMFI codes must be valid 6-digit numeric codes.


In [3]:
def validate_amfi(code):
    try:
        val = int(code)
        return 100000 <= val <= 999999
    except (ValueError, TypeError):
        return False

invalid_funds = df_fund[~df_fund['amfi_code'].apply(validate_amfi)]
print(f'Number of invalid AMFI codes in master: {len(invalid_funds)}')


Number of invalid AMFI codes in master: 0


## API Fetch using Requests: Fetching Live NAV from AMFI
We retrieve live data for `119551` (Aditya Birla Sun Life Banking & PSU Debt Fund) from the open public API.


In [4]:
url = 'https://api.mfapi.in/mf/119551'
try:
    res = requests.get(url, timeout=10)
    if res.status_code == 200:
        data = res.json()
        meta = data.get('meta', {})
        nav_list = data.get('data', [])
        print('SUCCESSFULLY FETCHED:')
        print(f'  Scheme Name: {meta.get("scheme_name")}')
        print(f'  Fund House: {meta.get("fund_house")}')
        if nav_list:
            print(f'  Latest NAV: {nav_list[0].get("nav")} (Date: {nav_list[0].get("date")})')
    else:
        print(f'API error, status code: {res.status_code}')
except Exception as e:
    print(f'Could not fetch live API data: {e}')


SUCCESSFULLY FETCHED:
  Scheme Name: Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - IDCW
  Fund House: Aditya Birla Sun Life Mutual Fund
  Latest NAV: 105.30840 (Date: 08-06-2026)
